# CS F425 Deep Learning Project — Phase 1
## QLoRA Fine-Tuning of Phi-2 for Structured Data Analysis

Extracted from `CS_F425_DL_Project.ipynb`.

Fine-tunes `microsoft/phi-2` (4-bit QLoRA) to act as an AI agent over `sales_data.csv`.  
Given a natural language query, the model outputs:
```json
{"actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"], "answer": 52345678.12}
```

**Runtime target**: ≤6 hours on Colab free-tier T4 GPU.


## Cell 1 — Install Packages

In [1]:
import os
import subprocess
import sys
from pathlib import Path

DEPS_SENTINEL = Path("/tmp/.csf425_phi2_deps_ready")
PACKAGES = [
    "transformers>=4.41.0",
    "peft>=0.10.0",
    "trl>=0.9.0",
    "accelerate>=0.29.3",
    "datasets>=2.19.0",
    "bitsandbytes>=0.44.0",
    "einops",
]

if not DEPS_SENTINEL.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *PACKAGES])
    DEPS_SENTINEL.write_text("ok")
    print("Dependencies installed or upgraded. Restarting the Colab runtime once...")
    os.kill(os.getpid(), 9)

print("Dependencies already prepared for this runtime.")


Dependencies already prepared for this runtime.


In [2]:
# Cell 1 restarts the Colab runtime once after upgrading packages.
# After the reconnect, continue from the next cell.


## Cell 2 — Imports & GPU Check

In [3]:
import json
import random
import re

import pandas as pd
import torch
from datasets import Dataset
from packaging import version
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

try:
    import trl
except Exception as exc:
    raise RuntimeError(
        "Failed to import trl. Re-run Cell 1 and let Colab restart once."
    ) from exc

if not hasattr(trl, "SFTTrainer"):
    raise RuntimeError(
        "trl.SFTTrainer is unavailable. Re-run Cell 1 and let Colab restart once."
    )

SFTTrainer = trl.SFTTrainer
SFTConfig = getattr(trl, "SFTConfig", TrainingArguments)
_USE_SFTCONFIG = hasattr(trl, "SFTConfig")

import transformers as _tf

if version.parse(_tf.__version__) < version.parse("4.41.0"):
    raise RuntimeError(
        f"transformers=={_tf.__version__} is too old for this notebook. "
        "Re-run Cell 1 and let Colab restart once."
    )

print(f"trl         : {trl.__version__}  (SFTConfig available: {_USE_SFTCONFIG})")
print(f"transformers: {_tf.__version__}")

assert torch.cuda.is_available(), "No GPU. In Colab, switch to a T4 GPU runtime."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


trl         : 1.2.0  (SFTConfig available: True)
transformers: 5.5.4
GPU : Tesla T4
VRAM: 15.6 GB


## Cell 3 — Mount Google Drive

Saves adapter weights and checkpoints to Drive so they persist across Colab sessions.

**One-time setup**: Upload these files to `MyDrive/CS_F425_Project/`:
- `sales_data.csv`
- `agent_trajectories_2k.json`
- `tool_executor.py`
- `run_pipeline.py`

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob as _glob
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Make tool_executor.py importable
sys.path.insert(0, DRIVE_DIR)


def detect_training_state():
    """Detect what checkpoints are available and decide the execution path.

    Returns (state, best_path) where state is one of:
      TRAINING_COMPLETE  - final adapter saved (skip training, go to inference)
      EPOCH_CHECKPOINT   - HF Trainer checkpoint exists (resume training)
      TIMED_CHECKPOINT   - timed adapter-only snapshot (skip training, inference-capable)
      FRESH              - nothing found (train from scratch)

    Checkpoints are sorted by their numeric suffix (highest number wins),
    so deleting earlier checkpoints to save space will not cause issues.
    """
    # 1. Final adapter saved by Cell 13?
    if os.path.isfile(f"{ADAPTER_DIR}/adapter_config.json"):
        return "TRAINING_COMPLETE", ADAPTER_DIR

    # 2. Trainer epoch checkpoints (contain full optimizer state)?
    epoch_ckpts = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]

    # 3. Timed adapter-only snapshots?
    timed_ckpts = _glob.glob(f"{ADAPTER_DIR}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]

    return "FRESH", None


TRAINING_STATE, BEST_CKPT_PATH = detect_training_state()
SKIP_TRAINING = TRAINING_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print(f"Drive dir       : {DRIVE_DIR}")
print(f"Contents        : {os.listdir(DRIVE_DIR)}")
print(f"Training state  : {TRAINING_STATE}")
print(f"Best checkpoint : {BEST_CKPT_PATH}")
print(f"Skip training   : {SKIP_TRAINING}")

Mounted at /content/drive
Drive dir       : /content/drive/MyDrive/CS_F425_Project
Contents        : ['agent_trajectories_2k.json', 'run_pipeline.py', 'tool_executor.py', 'sales_data.csv', 'phi2-agent-adapter', 'phi2-agent-qlora', '__pycache__', 'mistral-react-adapter_divyam', 'mistral-react-qlora_divyam', 'mistral-react-adapter-v2', 'mistral-react-qlora-v2', 'mistral-react-adapter-fast', 'mistral-react-qlora-fast', 'mistral-react-adapter-fast-nogc', 'mistral-react-qlora-fast-nogc', 'mistral-react-toolalpaca-warmup-fast-nogc', 'mistral-react-toolalpaca-warmup-ckpt-fast-nogc', 'toolalpaca_train_data.json', 'mistral-react-dpo-adapter-fast-nogc', 'mistral-react-dpo-ckpt-fast-nogc', 'mistral-react-adapter_divyam_duplicate', 'mistral-react-qlora_divyam_duplicate', 'mistral-react-adapter_divyam_duplicate2', 'mistral-react-qlora_divyam_duplicate2', 'mistral-react-dpo-adapter_divyam_duplicate2', 'mistral-react-dpo-ckpt_divyam_duplicate2', 'mistral-react-dpo-ckpt-222', 'tool_alpaca_cache.json',

## Cell 4 — Load Data

In [5]:
df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
print(f"sales_data shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

with open(f"{DRIVE_DIR}/agent_trajectories_2k.json") as f:
    trajectories = json.load(f)
print(f"\nTrajectories loaded: {len(trajectories)}")
print("Sample entry:", json.dumps(trajectories[0], indent=2))

sales_data shape: (10000, 11)
date           object
year            int64
month           int64
city           object
region         object
product        object
category       object
revenue       float64
units_sold      int64
cost          float64
profit        float64
dtype: object
         date  year  month     city region product     category   revenue  \
0  2023-12-06  2023     12  Kolkata   East       D  Electronics  15302.75   
1  2022-04-12  2022      4    Delhi  North       D  Electronics  17626.77   
2  2022-12-01  2022     12  Chennai  South       B     Clothing  10111.14   

   units_sold      cost   profit  
0          60   8177.77  7124.98  
1          32  11693.71  5933.06  
2          53   7306.30  2804.85  

Trajectories loaded: 2000
Sample entry: {
  "query": "What is total revenue for 2021?",
  "actions": [
    "filter_data(column='year', value=2021)",
    "aggregate_sum(column='revenue')"
  ]
}


## Cell 5 — Pre-compute Answers

The training data has `query` + `actions` but **no answers**.  
We execute each action sequence against `sales_data.csv` using `ToolExecutor` to obtain the gold answer.

We also extend `parse_agent_action` to handle `aggregate_mean`, `aggregate_count`, and string filter values that the original `run_pipeline.py` misses.

In [6]:
from numbers import Integral, Real

from tool_executor import ToolExecutor


def _parse_numeric(val_str):
    """Convert a string to int or float as appropriate."""
    return float(val_str) if '.' in val_str else int(val_str)


def parse_agent_action(action_str):
    """Parse one string-form action into ToolExecutor format."""
    if action_str.startswith("filter_data"):
        # Try numeric value first (int or float, possibly negative)
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.[\d]+)?)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))},
            }

        # Try string value
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    """Convert ToolExecutor output into the JSON shape used for supervision."""
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None

    action_names = [action.split("(")[0] for action in actions]

    # Single scalar result
    if result_df.shape == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    # group_by + aggregate (without sort/topk) → key-value dict
    has_groupby = any(a.startswith("group_by") for a in action_names)
    has_sort_or_topk = any(
        a.startswith("sort_by") or a.startswith("top_k") for a in action_names
    )
    if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
        key_col, value_col = result_df.columns
        return {
            str(row[key_col]): clean_scalar(row[value_col])
            for _, row in result_df.iterrows()
        }

    # Default: list of dicts (for sorted/topk results or multi-column outputs)
    return [
        {str(key): clean_scalar(value) for key, value in row.items()}
        for row in result_df.to_dict(orient="records")
    ]


def compute_answer(actions, df):
    """Parse and execute a list of action strings."""
    parsed = []
    for action in actions:
        try:
            parsed_action = parse_agent_action(action)
        except Exception:
            parsed_action = None

        if parsed_action is not None:
            parsed.append(parsed_action)

    if not parsed:
        return None

    try:
        result = ToolExecutor(df.copy()).execute(parsed)
        return result_to_python(actions, result)
    except Exception:
        return None


training_data = []
skipped = 0

for entry in trajectories:
    answer = compute_answer(entry["actions"], df)
    if answer is None:
        skipped += 1
        continue

    output_json = json.dumps(
        {"actions": entry["actions"], "answer": answer},
        ensure_ascii=False,
    )
    training_data.append({"query": entry["query"], "output_json": output_json})

print(f"Valid examples : {len(training_data)} / {len(trajectories)}")
print(f"Skipped        : {skipped}")
print("\nSample:")
print(json.dumps(json.loads(training_data[0]["output_json"]), indent=2, ensure_ascii=False))

Valid examples : 2000 / 2000
Skipped        : 0

Sample:
{
  "actions": [
    "filter_data(column='year', value=2021)",
    "aggregate_sum(column='revenue')"
  ],
  "answer": 44882702.02
}


## Cell 6 — Schema String & Prompt Template

The prompt follows the same `### Task / ### Schema / ### Question / ### Answer` pattern as Lab02.  
The schema string is fixed — it is identical at training and inference time.

In [7]:
SCHEMA = """Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)"""

PROMPT_TEMPLATE = """### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an \"actions\" list and an \"answer\" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
"""

MAX_SEQ_LENGTH = 384


def make_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)


EOS = None  # Set properly in Cell 8 after tokenizer loads


def format_example(row, eos_token=None):
    eos = eos_token or EOS
    if eos is None:
        raise ValueError("EOS token not set. Run Cell 8 (tokenizer) before Cell 9.")
    return {"text": make_prompt(row["query"]) + row["output_json"] + eos}


print(make_prompt(training_data[0]["query"]))
print(f"Configured max sequence length: {MAX_SEQ_LENGTH}")

### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an "actions" list and an "answer" field. No other text.

### Schema
Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)

### Question
What is total revenue for 2021?

### Answer

Configured max sequence length: 384


## Cell 7 — Split Dataset

In [8]:
random.seed(42)
random.shuffle(training_data)

TRAIN_SIZE = min(1800, int(len(training_data) * 0.9))
VALID_SIZE = 200

raw_train = training_data[:TRAIN_SIZE]
raw_valid = training_data[TRAIN_SIZE : TRAIN_SIZE + VALID_SIZE]

print(f"Train : {len(raw_train)}")
print(f"Valid : {len(raw_valid)}")

Train : 1800
Valid : 200


## Cell 8 — Load Tokenizer

In [9]:
MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
EOS = tokenizer.eos_token

print(f"EOS token: {EOS!r}")

sample_text = make_prompt(raw_train[0]["query"]) + raw_train[0]["output_json"] + EOS
n_tokens = len(tokenizer(sample_text).input_ids)
print(f"Sample token length: {n_tokens} (target <= {MAX_SEQ_LENGTH})")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

EOS token: '<|endoftext|>'
Sample token length: 338 (target <= 384)


## Cell 9 — Build HuggingFace Datasets

In [10]:
def fmt(row):
    return format_example(row, eos_token=tokenizer.eos_token)

train_ds = Dataset.from_list(raw_train).map(
    fmt, remove_columns=["query", "output_json"]
)
valid_ds = Dataset.from_list(raw_valid).map(
    fmt, remove_columns=["query", "output_json"]
)

print(f"Train dataset : {len(train_ds)} examples")
print(f"Valid dataset : {len(valid_ds)} examples")
print("\nSample text (truncated):")
print(train_ds[0]["text"][:500])

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Train dataset : 1800 examples
Valid dataset : 200 examples

Sample text (truncated):
### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an "actions" list and an "answer" field. No other text.

### Schema
Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  a


## Cell 10 — Load Phi-2 in 4-bit (QLoRA)

Config mirrors Lab02 exactly: NF4 double-quant, bfloat16 compute dtype, gradient checkpointing.

In [11]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

if SKIP_TRAINING:
    # ── Inference-only path: load base model + saved adapter ──
    from peft import PeftModel

    print(f"Loading base model + saved adapter from: {BEST_CKPT_PATH}")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, BEST_CKPT_PATH)
    model.eval()
    model.config.use_cache = True
    print("Adapter loaded — model ready for inference.")
else:
    # ── Training path: load base model + prepare for k-bit training ──
    print("Loading base model for training...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    print("Base model loaded and prepared for QLoRA training.")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

Compute dtype: torch.bfloat16
Loading base model + saved adapter from: /content/drive/MyDrive/CS_F425_Project/phi2-agent-adapter


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Adapter loaded — model ready for inference.
2119 MiB, 15360 MiB


## Cell 11 — Apply LoRA Adapters

In [12]:
if not SKIP_TRAINING:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Expected: ~0.84% trainable (~23M of 2.8B params)
else:
    print("Skipping LoRA setup — adapter already loaded.")
    model.print_trainable_parameters()

Skipping LoRA setup — adapter already loaded.
trainable params: 0 || all params: 2,803,276,800 || trainable%: 0.0000


## Cell 12 — Fine-Tune with SFTTrainer

Config mirrors Lab02: cosine LR, paged AdamW-8bit, effective batch 16, 2 epochs.  
Checkpoints saved to Drive so training can resume on Colab disconnect.

In [13]:
if SKIP_TRAINING:
    print(f"Training already complete. Adapter loaded from: {BEST_CKPT_PATH}")
    print("Skipping training — proceed to inference cells below.")
else:
    import gc
    import inspect as _inspect
    import glob as _glob
    import time as _time
    from transformers import TrainerCallback

    _sft_sig = set(_inspect.signature(SFTConfig.__init__).parameters.keys())
    _trainer_sig = set(_inspect.signature(SFTTrainer.__init__).parameters.keys())

    _eval_key = "eval_strategy" if "eval_strategy" in _sft_sig else "evaluation_strategy"
    _tok_key = "processing_class" if "processing_class" in _trainer_sig else "tokenizer"

    _steps_per_epoch = max(1, len(train_ds) // (4 * 4))
    _warmup_steps = max(1, int(0.05 * _steps_per_epoch * 2))
    _total_steps = _steps_per_epoch * 2  # num_train_epochs=2

    # --- Find the highest checkpoint by step number ---
    _checkpoints = _glob.glob(f"{CKPT_DIR}/checkpoint-*")
    _resume_ckpt = None
    _training_already_done = False

    if _checkpoints:
        _checkpoints.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        _resume_ckpt = _checkpoints[-1]
        _latest_step = int(_resume_ckpt.rsplit("-", 1)[-1])
        print(f"Found {len(_checkpoints)} checkpoint(s). Highest: step {_latest_step} / {_total_steps}")

        if _latest_step >= _total_steps:
            _training_already_done = True
            print(f"Latest checkpoint (step {_latest_step}) >= total steps ({_total_steps}).")
            print(f"Training already complete — loading adapter from: {_resume_ckpt}")
            # Reload model with trained adapter from the checkpoint
            from peft import PeftModel
            del model
            gc.collect()
            torch.cuda.empty_cache()
            _base = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            model = PeftModel.from_pretrained(_base, _resume_ckpt)
            model.eval()
            model.config.use_cache = True
            SKIP_TRAINING = True
        else:
            print(f"Resuming training from checkpoint: {_resume_ckpt} (step {_latest_step}/{_total_steps})")
    else:
        print("No checkpoint found — starting fresh.")

    if not _training_already_done:
        _cfg_extra = {}
        _trainer_extra = {}
        for _param, _val in [
            ("max_seq_length", MAX_SEQ_LENGTH),
            ("dataset_text_field", "text"),
            ("packing", False),
        ]:
            if _param in _sft_sig:
                _cfg_extra[_param] = _val
            elif _param in _trainer_sig:
                _trainer_extra[_param] = _val

        _common_args = dict(
            output_dir=CKPT_DIR,
            seed=42,
            num_train_epochs=2,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            lr_scheduler_type="cosine",
            warmup_steps=_warmup_steps,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=50,
            save_strategy="epoch",
            save_total_limit=3,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            report_to="none",
        )

        training_args = SFTConfig(
            **_common_args,
            **{_eval_key: "epoch"},
            **_cfg_extra,
        )

        class TimedCheckpointCallback(TrainerCallback):
            """Saves adapter weights to Drive every `interval_min` minutes."""

            def __init__(self, adapter_dir, interval_min=5):
                self.adapter_dir = adapter_dir
                self.interval_sec = interval_min * 60
                self.last_save = _time.time()

            def on_step_end(self, args, state, control, model=None, **kwargs):
                elapsed = _time.time() - self.last_save
                if elapsed >= self.interval_sec:
                    save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                    os.makedirs(save_path, exist_ok=True)
                    model.save_pretrained(save_path)
                    tokenizer.save_pretrained(save_path)
                    self.last_save = _time.time()
                    print(f"\n[TimedCheckpoint] Saved adapter at step {state.global_step} "
                          f"to {save_path} ({elapsed/60:.1f} min since last save)")

        trainer = SFTTrainer(
            model=model,
            **{_tok_key: tokenizer},
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=valid_ds,
            callbacks=[TimedCheckpointCallback(ADAPTER_DIR, interval_min=5)],
            **_trainer_extra,
        )

        print(f"Training on {len(train_ds)} examples, validating on {len(valid_ds)}")
        print(f"Epochs: 2  |  Effective batch: 16  |  Warmup steps: {_warmup_steps}")
        print(f"Max sequence length: {MAX_SEQ_LENGTH}")
        print(f"SFT params -> SFTConfig: {_cfg_extra}  |  SFTTrainer: {_trainer_extra}")
        print(f"Tokenizer key: {_tok_key!r}")

        trainer.train(resume_from_checkpoint=_resume_ckpt)

Training already complete. Adapter loaded from: /content/drive/MyDrive/CS_F425_Project/phi2-agent-adapter
Skipping training — proceed to inference cells below.


## Cell 13 — Save Adapter Weights to Drive

In [14]:
if not SKIP_TRAINING:
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to: {ADAPTER_DIR}")
else:
    print(f"Adapter was already saved at: {BEST_CKPT_PATH}")

print("\nAdapter directory contents:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    fpath = os.path.join(ADAPTER_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {fname:<40s}  {size_mb:.2f} MB")
    else:
        print(f"  {fname:<40s}  [directory]")

Adapter was already saved at: /content/drive/MyDrive/CS_F425_Project/phi2-agent-adapter

Adapter directory contents:
  README.md                                 0.01 MB
  adapter_config.json                       0.00 MB
  adapter_model.safetensors                 94.42 MB
  timed_ckpt_step_105                       [directory]
  timed_ckpt_step_112                       [directory]
  timed_ckpt_step_115                       [directory]
  timed_ckpt_step_122                       [directory]
  timed_ckpt_step_129                       [directory]
  timed_ckpt_step_136                       [directory]
  timed_ckpt_step_14                        [directory]
  timed_ckpt_step_143                       [directory]
  timed_ckpt_step_150                       [directory]
  timed_ckpt_step_157                       [directory]
  timed_ckpt_step_164                       [directory]
  timed_ckpt_step_171                       [directory]
  timed_ckpt_step_178                       [directory

## Cell 14 — Inference Function

Given a natural language question, generates the JSON output and parses it back to a Python dict.

In [15]:
def generate_answer(question: str, max_new_tokens: int = 200) -> dict:
    """Run the fine-tuned model on a question and return a parsed JSON dict."""
    model.eval()
    model.config.use_cache = True

    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return {"raw_output": raw}


## Cell 15 — Evaluation on Held-Out Queries

Runs the model on a set of test questions and cross-checks the model's answer against
the ground-truth answer computed directly by `ToolExecutor`.

In [16]:
def normalize_answer(value):
    if isinstance(value, float):
        return round(value, 4)
    if isinstance(value, list):
        return [normalize_answer(item) for item in value]
    if isinstance(value, dict):
        return {key: normalize_answer(val) for key, val in value.items()}
    return value


test_queries = [
    {
        "question": "What is the total revenue for 2022?",
        "expected_actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"],
    },
    {
        "question": "Which city had the highest profit in 2021? Top 1",
        "expected_actions": ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", "top_k(k=1)"],
    },
    {
        "question": "What is the average revenue by city?",
        "expected_actions": ["group_by(column='city')", "aggregate_mean(column='revenue')"],
    },
    {
        "question": "List top 3 cities by revenue in 2022.",
        "expected_actions": ["filter_data(column='year', value=2022)", "group_by(column='city')", "aggregate_sum(column='revenue')", "sort_by(column='revenue', order='desc')", "top_k(k=3)"],
    },
    {
        "question": "What is the total profit for 2023?",
        "expected_actions": ["filter_data(column='year', value=2023)", "aggregate_sum(column='profit')"],
    },
]

for entry in test_queries:
    q = entry["question"]
    expected_actions = entry["expected_actions"]
    expected_answer = compute_answer(expected_actions, df)

    result = generate_answer(q)
    model_actions = result.get("actions", [])
    model_answer = compute_answer(model_actions, df) if model_actions else None

    print(f"Q: {q}")
    print(f"  Expected actions : {expected_actions}")
    print(f"  Model actions    : {model_actions}")
    print(f"  Expected answer  : {expected_answer}")
    print(f"  Model answer     : {model_answer}")
    print(f"  Match            : {normalize_answer(model_answer) == normalize_answer(expected_answer)}")
    print()

Q: What is the total revenue for 2022?
  Expected actions : ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"]
  Model actions    : ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"]
  Expected answer  : 44926739.48
  Model answer     : 44926739.48
  Match            : True

Q: Which city had the highest profit in 2021? Top 1
  Expected actions : ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", 'top_k(k=1)']
  Model actions    : ["filter_data(column='year', value=2021)", "group_by(column='city')", "aggregate_sum(column='profit')", "sort_by(column='profit', order='desc')", 'top_k(k=1)']
  Expected answer  : [{'city': 'Mumbai', 'profit': 3730922.48}]
  Model answer     : [{'city': 'Mumbai', 'profit': 3730922.48}]
  Match            : True

Q: What is the average revenue by city?
  Expected actions : ["group_by(column='city')", "aggregate_m